# Amazon Search / LTR Project — Week 3
## LLM-Powered Query Understanding — One-Click Notebook

This notebook implements:

**Week 1/2 data → Llama 3 via Ollama → structured JSON → intent classification → attribute extraction → query rewriting → BM25 A/B comparison → evaluation CSVs**

### Upload these two files when prompted
- `amazon_week1_v4_processed_data.zip`
- `amazon_week2_baseline_ranking_results.zip`

### Default experiment
- Model: `llama3:8b`
- Start with 50 fixed Week-2 test queries
- Expand to 200 after the first run is stable
- Temperature = 0
- Strong relevance for full-catalog recall: `relevance_label >= 2`

### Main outputs
- `week3_structured_queries.csv`
- `week3_structured_queries.parquet`
- `week3_manual_validation_50.csv`
- `week3_bm25_rewrite_evaluation_per_query.csv`
- `week3_bm25_rewrite_summary.csv`
- `week3_query_understanding_summary.csv`
- `amazon_week3_results.zip`

**Why two kinds of retrieval evaluation?**  
Week 2 Recall@10 was saturated on the small judged candidate sets. This notebook therefore keeps graded NDCG/MRR on judged candidates, while also retrieving from the full 10k-product catalog and measuring recovery of known strong positives.

# Day 1 — Local LLM Setup & Structured Output

**Goal:** install/test Ollama + Llama 3, load the existing Week 1/2 artifacts,
call the local model from Python, and verify that it returns valid JSON.

**Day-1 completion standard:** `Python → Ollama → Llama 3 → valid JSON`.

## 1. Install packages

In [ ]:
!pip -q install pandas numpy pyarrow rank-bm25 scikit-learn tqdm requests matplotlib

## 2. Start Ollama and pull Llama 3 8B

The first pull can take several minutes because the model is several GB. A GPU runtime is recommended.

For the formal Week-3 benchmark keep `MODEL = "llama3:8b"`.

In [ ]:
import os
import time
import shutil
import subprocess
import requests

MODEL = "llama3:8b"
OLLAMA_BASE_URL = "http://127.0.0.1:11434"

def ollama_alive():
    try:
        r = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=2)
        return r.ok
    except Exception:
        return False

if shutil.which("ollama") is None:
    print("Ollama not found. Installing...")
    subprocess.run(
        "curl -fsSL https://ollama.com/install.sh | sh",
        shell=True,
        check=True
    )

if not ollama_alive():
    print("Starting Ollama server...")
    log_file = open("/tmp/ollama_week3.log", "w")
    OLLAMA_PROCESS = subprocess.Popen(
        ["ollama", "serve"],
        stdout=log_file,
        stderr=subprocess.STDOUT
    )
    for _ in range(60):
        if ollama_alive():
            break
        time.sleep(1)

assert ollama_alive(), "Ollama did not start. Check /tmp/ollama_week3.log."

print(f"Pulling/checking model: {MODEL}")
subprocess.run(["ollama", "pull", MODEL], check=True)
print("Ollama is ready.")

## 3. Upload and extract Week 1 + Week 2 ZIPs

In Colab, select both files in the upload dialog.  
If running locally, place both ZIPs in the current working directory first.

In [ ]:
from pathlib import Path
import zipfile

EXPECTED_ZIPS = [
    "amazon_week1_v4_processed_data.zip",
    "amazon_week2_baseline_ranking_results.zip",
]

def running_in_colab():
    try:
        import google.colab
        return True
    except Exception:
        return False

missing = [f for f in EXPECTED_ZIPS if not Path(f).exists()]

if missing and running_in_colab():
    from google.colab import files
    print("Upload BOTH Week 1 and Week 2 ZIP files.")
    files.upload()

missing = [f for f in EXPECTED_ZIPS if not Path(f).exists()]
if missing:
    raise FileNotFoundError("Missing: " + ", ".join(missing))

WORKDIR = Path("amazon_week3_work")
INPUT_DIR = WORKDIR / "inputs"
RESULTS_DIR = WORKDIR / "results"
INPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

for zip_name in EXPECTED_ZIPS:
    target = INPUT_DIR / Path(zip_name).stem
    target.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_name, "r") as z:
        z.extractall(target)
    print("Extracted:", zip_name, "->", target)

## 4. Load data and preserve the fixed Week-2 test split

In [ ]:
import pandas as pd
import numpy as np
import json
import re
from IPython.display import display

W1_DIR = INPUT_DIR / "amazon_week1_v4_processed_data"
W2_DIR = INPUT_DIR / "amazon_week2_baseline_ranking_results"

qpr = pd.read_csv(W1_DIR / "query_product_relevance.csv")
products = pd.read_parquet(W1_DIR / "cleaned_products_10k.parquet")
test_ids = pd.read_csv(W2_DIR / "fixed_test_queries.csv")
week2_metrics = pd.read_csv(W2_DIR / "benchmark_metrics.csv")

qpr["query_id"] = qpr["query_id"].astype(str)
qpr["product_id"] = qpr["product_id"].astype(str)
products["product_id"] = products["product_id"].astype(str)
test_ids["query_id"] = test_ids["query_id"].astype(str)

query_table = (
    qpr[["query_id", "query"]]
    .drop_duplicates("query_id")
    .merge(test_ids, on="query_id", how="inner")
)

order_map = {qid: i for i, qid in enumerate(test_ids["query_id"].tolist())}
query_table["_order"] = query_table["query_id"].map(order_map)
query_table = (
    query_table
    .sort_values("_order")
    .drop(columns="_order")
    .reset_index(drop=True)
)

print("Query-product pairs:", len(qpr))
print("Catalog products:", len(products))
print("Fixed test queries:", len(query_table))
print()
display(week2_metrics)
display(query_table.head(10))

# Day 2 — Intent Classification & Core Product Extraction

**Goal:** define a stable e-commerce intent taxonomy and prompt, then extract:
`intent`, `core_product`, `brand`, and `category`.

The same structured schema is intentionally reused through the week so later
evaluation is reproducible.

## 5. Structured JSON schema + prompt

One LLM call performs:
- intent classification
- core product extraction
- brand/category extraction
- attribute extraction
- price extraction
- use-case extraction
- implicit requirement extraction
- conservative query rewriting

The prompt tells the model not to invent unsupported product facts.

In [ ]:
QUERY_SCHEMA = {
    "type": "object",
    "properties": {
        "query": {"type": "string"},
        "intent": {
            "type": "string",
            "enum": [
                "exact_product_search",
                "product_search",
                "category_browse",
                "brand_search",
                "use_case_search",
                "compatibility_search",
                "other"
            ]
        },
        "core_product": {"type": ["string", "null"]},
        "brand": {"type": ["string", "null"]},
        "category": {"type": ["string", "null"]},
        "attributes": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "name": {"type": "string"},
                    "value": {"type": "string"},
                    "explicit": {"type": "boolean"}
                },
                "required": ["name", "value", "explicit"],
                "additionalProperties": False
            }
        },
        "price_min": {"type": ["number", "null"]},
        "price_max": {"type": ["number", "null"]},
        "use_case": {"type": ["string", "null"]},
        "implicit_requirements": {
            "type": "array",
            "items": {"type": "string"}
        },
        "rewritten_query": {"type": "string"}
    },
    "required": [
        "query", "intent", "core_product", "brand", "category",
        "attributes", "price_min", "price_max", "use_case",
        "implicit_requirements", "rewritten_query"
    ],
    "additionalProperties": False
}

SYSTEM_PROMPT = """
You are the query-understanding component of an e-commerce product search system.

For every query, return one structured JSON object following the supplied schema.

TASKS
1. Classify the search intent.
2. Extract the core product if stated or identifiable with high confidence.
3. Extract explicit brand and category information.
4. Extract product attributes such as color, size, material, gender, compatibility,
   feature, quantity, style, condition, or other constraints.
5. Extract numeric price_min / price_max only when a real price constraint is present.
6. Extract the use case.
7. Infer only high-confidence implicit requirements.
8. Rewrite the query into a concise retrieval-friendly product-search query.

RULES
- Be conservative.
- Do not invent a brand, size, color, model number, price, or product type.
- If unknown, use null or an empty list.
- Preserve important explicit terms.
- Remove conversational filler from the rewrite.
- Add an implicit term only when strongly implied.
- Do not recommend products.
- Do not explain.
- Output JSON only.
""".strip()

print(SYSTEM_PROMPT)

## 6. Ollama inference function

It first tries schema-constrained structured output and falls back to normal JSON mode if needed.

In [ ]:
import requests
import json
import time

REQUIRED_KEYS = list(QUERY_SCHEMA["required"])

def validate_result(obj, original_query):
    if not isinstance(obj, dict):
        raise ValueError("Output is not a JSON object.")
    missing = [k for k in REQUIRED_KEYS if k not in obj]
    if missing:
        raise ValueError(f"Missing keys: {missing}")

    obj["query"] = original_query

    if not isinstance(obj.get("attributes"), list):
        obj["attributes"] = []
    if not isinstance(obj.get("implicit_requirements"), list):
        obj["implicit_requirements"] = []

    return obj

def understand_query(query, model=MODEL, max_retries=3):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Query: {query}"}
    ]

    last_error = None

    for attempt in range(1, max_retries + 1):
        start = time.perf_counter()

        payload = {
            "model": model,
            "messages": messages,
            "stream": False,
            "format": QUERY_SCHEMA,
            "options": {
                "temperature": 0,
                "num_ctx": 4096
            }
        }

        try:
            r = requests.post(
                f"{OLLAMA_BASE_URL}/api/chat",
                json=payload,
                timeout=180
            )

            if r.status_code >= 400:
                payload["format"] = "json"
                r = requests.post(
                    f"{OLLAMA_BASE_URL}/api/chat",
                    json=payload,
                    timeout=180
                )

            r.raise_for_status()
            raw_text = r.json()["message"]["content"]
            obj = json.loads(raw_text)
            obj = validate_result(obj, query)

            latency = time.perf_counter() - start
            return obj, latency, raw_text

        except Exception as e:
            last_error = repr(e)
            time.sleep(min(attempt * 2, 5))

    raise RuntimeError(
        f"LLM query understanding failed after {max_retries} tries: {last_error}"
    )

## 7. Smoke test — Day 1 completion check

In [ ]:
demo_query = "black waterproof hiking shoes for women under $100"

demo_result, demo_latency, _ = understand_query(demo_query)

print("Latency:", round(demo_latency, 2), "seconds")
print(json.dumps(demo_result, indent=2, ensure_ascii=False))

# Day 3 — Attribute Extraction & Structured Query Dataset

**Goal:** batch the fixed Week-2 test queries and extract:
price, color/feature attributes, use case, compatibility, brand/category,
implicit requirements, and a rewritten query.

Start with **50 queries**. After the manual validation is stable, change
`N_QUERIES = 200` and rerun for the formal Week-3 benchmark.

## 8. Batch query understanding

Start with `N_QUERIES = 50`.  
When stable, change it to `200`.

A checkpoint is saved after every query, so the run can resume after interruption.

In [ ]:
from tqdm.auto import tqdm

N_QUERIES = 50
CHECKPOINT_PATH = RESULTS_DIR / "week3_llm_outputs_checkpoint.csv"

run_queries = query_table.head(N_QUERIES).copy()

if CHECKPOINT_PATH.exists():
    existing = pd.read_csv(CHECKPOINT_PATH)
    existing["query_id"] = existing["query_id"].astype(str)
else:
    existing = pd.DataFrame()

completed_ids = set(existing["query_id"]) if len(existing) else set()
new_rows = []

def serialize_result(obj, query_id, latency, status="ok", error=None):
    return {
        "query_id": str(query_id),
        "query": obj.get("query"),
        "intent": obj.get("intent"),
        "core_product": obj.get("core_product"),
        "brand": obj.get("brand"),
        "category": obj.get("category"),
        "attributes_json": json.dumps(obj.get("attributes", []), ensure_ascii=False),
        "price_min": obj.get("price_min"),
        "price_max": obj.get("price_max"),
        "use_case": obj.get("use_case"),
        "implicit_requirements_json": json.dumps(
            obj.get("implicit_requirements", []),
            ensure_ascii=False
        ),
        "rewritten_query": obj.get("rewritten_query"),
        "latency_sec": latency,
        "parse_status": status,
        "error": error
    }

for row in tqdm(run_queries.itertuples(index=False), total=len(run_queries)):
    if row.query_id in completed_ids:
        continue

    try:
        result, latency, raw = understand_query(row.query)
        out = serialize_result(result, row.query_id, latency)
    except Exception as e:
        out = {
            "query_id": str(row.query_id),
            "query": row.query,
            "intent": None,
            "core_product": None,
            "brand": None,
            "category": None,
            "attributes_json": "[]",
            "price_min": None,
            "price_max": None,
            "use_case": None,
            "implicit_requirements_json": "[]",
            "rewritten_query": None,
            "latency_sec": np.nan,
            "parse_status": "error",
            "error": repr(e)
        }

    new_rows.append(out)

    current = pd.concat(
        [existing, pd.DataFrame(new_rows)],
        ignore_index=True
    )
    current.to_csv(CHECKPOINT_PATH, index=False)

structured = pd.read_csv(CHECKPOINT_PATH)
structured["query_id"] = structured["query_id"].astype(str)
structured = structured[
    structured["query_id"].isin(run_queries["query_id"])
].copy()

structured["_order"] = structured["query_id"].map(
    {qid: i for i, qid in enumerate(run_queries["query_id"].tolist())}
)
structured = (
    structured
    .sort_values("_order")
    .drop(columns="_order")
    .reset_index(drop=True)
)

structured.to_csv(
    RESULTS_DIR / "week3_structured_queries.csv",
    index=False
)
structured.to_parquet(
    RESULTS_DIR / "week3_structured_queries.parquet",
    index=False
)

display(structured.head(10))

print("Queries processed:", len(structured))
print(
    "Parse success rate:",
    round((structured["parse_status"] == "ok").mean(), 4)
)
print(
    "Mean latency:",
    round(
        structured.loc[
            structured["parse_status"] == "ok",
            "latency_sec"
        ].mean(),
        2
    ),
    "sec/query"
)

## 9. Automatic diagnostics

In [ ]:
auto_summary = pd.DataFrame({
    "metric": [
        "queries_attempted",
        "parse_success_rate",
        "mean_latency_sec",
        "median_latency_sec",
        "core_product_non_null_rate",
        "rewrite_non_null_rate"
    ],
    "value": [
        len(structured),
        (structured["parse_status"] == "ok").mean(),
        structured["latency_sec"].mean(),
        structured["latency_sec"].median(),
        structured["core_product"].notna().mean(),
        structured["rewritten_query"].notna().mean()
    ]
})

auto_summary.to_csv(
    RESULTS_DIR / "week3_query_understanding_summary.csv",
    index=False
)

display(auto_summary)

print("Intent distribution:")
display(
    structured["intent"]
    .value_counts(dropna=False)
    .rename_axis("intent")
    .reset_index(name="count")
)

## 10. Create 50-query manual-validation template

Fill the `true_*` columns yourself.

Example `true_attributes_json`:

```json
{"color":"black","waterproof":"true","gender":"women"}
```

In [ ]:
MANUAL_N = min(50, len(structured))

manual = structured.head(MANUAL_N)[[
    "query_id",
    "query",
    "intent",
    "core_product",
    "brand",
    "category",
    "attributes_json",
    "price_min",
    "price_max",
    "use_case",
    "implicit_requirements_json",
    "rewritten_query"
]].copy()

manual = manual.rename(columns={
    "intent": "pred_intent",
    "core_product": "pred_core_product",
    "brand": "pred_brand",
    "category": "pred_category",
    "attributes_json": "pred_attributes_json",
    "price_min": "pred_price_min",
    "price_max": "pred_price_max",
    "use_case": "pred_use_case",
    "implicit_requirements_json": "pred_implicit_requirements_json",
    "rewritten_query": "pred_rewritten_query"
})

manual["true_intent"] = ""
manual["true_core_product"] = ""
manual["true_brand"] = ""
manual["true_category"] = ""
manual["true_attributes_json"] = ""
manual["true_price_min"] = ""
manual["true_price_max"] = ""
manual["true_use_case"] = ""
manual["rewrite_quality_0_to_2"] = ""
manual["notes"] = ""

manual_path = RESULTS_DIR / "week3_manual_validation_50.csv"
manual.to_csv(manual_path, index=False)

print("Created:", manual_path)
display(manual.head())

# Day 4 — BM25 Original Query vs Rewritten Query

### Judged-set metrics
- graded NDCG@10
- MRR

### Full 10k-catalog metric
Known strong positives are `relevance_label >= 2`:
- Recall@10
- Recall@50

This avoids claiming that all unjudged catalog products are irrelevant.

In [ ]:
from rank_bm25 import BM25Okapi
from sklearn.metrics import ndcg_score
from tqdm.auto import tqdm
import numpy as np
import re

def tokenize(text):
    if pd.isna(text):
        return []
    return re.findall(r"[a-z0-9]+", str(text).lower())

products["title"] = products["title"].fillna("").astype(str)
products["description"] = products["description"].fillna("").astype(str)
products["bm25_document"] = (
    products["title"].str.strip()
    + " "
    + products["description"].str.strip()
).str.strip()

tokenized_corpus = [
    tokenize(x)
    for x in tqdm(
        products["bm25_document"],
        desc="Tokenizing 10k-product catalog"
    )
]

bm25 = BM25Okapi(tokenized_corpus)

product_ids = products["product_id"].tolist()
pid_to_idx = {
    pid: i
    for i, pid in enumerate(product_ids)
}

print("BM25 catalog index built:", len(product_ids), "products")

In [ ]:
def reciprocal_rank(labels_ranked, relevant_threshold=1):
    for i, label in enumerate(labels_ranked, start=1):
        if label >= relevant_threshold:
            return 1.0 / i
    return 0.0

def recall_at_k(retrieved_ids, relevant_ids, k):
    relevant_ids = set(relevant_ids)
    if not relevant_ids:
        return np.nan
    return len(
        set(retrieved_ids[:k]) & relevant_ids
    ) / len(relevant_ids)

def bm25_scores_for_query(query):
    return np.asarray(
        bm25.get_scores(tokenize(query))
    )

def evaluate_one_version(query_id, query_text, scores, version_name):
    judged = qpr[
        qpr["query_id"] == str(query_id)
    ].copy()

    judged["catalog_idx"] = judged["product_id"].map(pid_to_idx)
    judged = judged[
        judged["catalog_idx"].notna()
    ].copy()

    judged["catalog_idx"] = judged["catalog_idx"].astype(int)
    judged["bm25_score_week3"] = scores[
        judged["catalog_idx"].values
    ]

    judged_ranked = judged.sort_values(
        "bm25_score_week3",
        ascending=False,
        kind="mergesort"
    )

    labels_ranked = (
        judged_ranked["relevance_label"]
        .astype(float)
        .tolist()
    )

    if len(judged) > 0:
        y_true = (
            judged["relevance_label"]
            .astype(float)
            .to_numpy()
            .reshape(1, -1)
        )
        y_score = (
            judged["bm25_score_week3"]
            .astype(float)
            .to_numpy()
            .reshape(1, -1)
        )
        ndcg10 = float(
            ndcg_score(y_true, y_score, k=10)
        )
    else:
        ndcg10 = np.nan

    mrr = reciprocal_rank(
        labels_ranked,
        relevant_threshold=1
    )

    max_k = min(50, len(scores))
    top_idx = np.argpartition(
        scores,
        -max_k
    )[-max_k:]

    top_idx = top_idx[
        np.argsort(scores[top_idx])[::-1]
    ]

    retrieved_ids = [
        product_ids[i]
        for i in top_idx
    ]

    strong_relevant_ids = set(
        judged.loc[
            judged["relevance_label"] >= 2,
            "product_id"
        ].astype(str)
    )

    return {
        "query_id": str(query_id),
        "version": version_name,
        "query_text": query_text,
        "judged_ndcg@10": ndcg10,
        "judged_mrr": mrr,
        "known_relevant_recall@10": recall_at_k(
            retrieved_ids,
            strong_relevant_ids,
            10
        ),
        "known_relevant_recall@50": recall_at_k(
            retrieved_ids,
            strong_relevant_ids,
            50
        ),
        "known_strong_relevant_count": len(
            strong_relevant_ids
        )
    }

eval_rows = []

usable = structured[
    (structured["parse_status"] == "ok")
    & structured["rewritten_query"].notna()
].copy()

for row in tqdm(
    usable.itertuples(index=False),
    total=len(usable),
    desc="BM25 original vs rewrite"
):
    original_scores = bm25_scores_for_query(
        row.query
    )
    rewritten_scores = bm25_scores_for_query(
        row.rewritten_query
    )

    eval_rows.append(
        evaluate_one_version(
            row.query_id,
            row.query,
            original_scores,
            "original"
        )
    )

    eval_rows.append(
        evaluate_one_version(
            row.query_id,
            row.rewritten_query,
            rewritten_scores,
            "rewritten"
        )
    )

eval_long = pd.DataFrame(eval_rows)

metric_cols = [
    "judged_ndcg@10",
    "judged_mrr",
    "known_relevant_recall@10",
    "known_relevant_recall@50"
]

summary = (
    eval_long
    .groupby("version")[metric_cols]
    .mean()
    .reset_index()
)

orig = summary[
    summary["version"] == "original"
].iloc[0]

rew = summary[
    summary["version"] == "rewritten"
].iloc[0]

delta_rows = []

for metric in metric_cols:
    original_value = float(orig[metric])
    rewritten_value = float(rew[metric])
    delta_abs = (
        rewritten_value - original_value
    )
    delta_pct = (
        delta_abs / original_value * 100
        if original_value != 0
        and not np.isnan(original_value)
        else np.nan
    )

    delta_rows.append({
        "metric": metric,
        "original": original_value,
        "rewritten": rewritten_value,
        "absolute_delta": delta_abs,
        "relative_delta_%": delta_pct
    })

summary_delta = pd.DataFrame(delta_rows)

wide = eval_long.pivot(
    index="query_id",
    columns="version",
    values=metric_cols
)

wide.columns = [
    f"{metric}_{version}"
    for metric, version in wide.columns
]

wide = wide.reset_index()

per_query = usable[
    ["query_id", "query", "rewritten_query"]
].merge(
    wide,
    on="query_id",
    how="left"
)

per_query["delta_ndcg@10"] = (
    per_query["judged_ndcg@10_rewritten"]
    - per_query["judged_ndcg@10_original"]
)

eval_long.to_csv(
    RESULTS_DIR / "week3_bm25_rewrite_evaluation_long.csv",
    index=False
)

per_query.to_csv(
    RESULTS_DIR / "week3_bm25_rewrite_evaluation_per_query.csv",
    index=False
)

summary_delta.to_csv(
    RESULTS_DIR / "week3_bm25_rewrite_summary.csv",
    index=False
)

print("Mean metrics:")
display(summary)

print("Original vs rewritten:")
display(summary_delta)

## 11. Plot and error analysis

In [ ]:
import matplotlib.pyplot as plt

plot_df = (
    summary_delta
    .set_index("metric")[["original", "rewritten"]]
)

ax = plot_df.plot(
    kind="bar",
    figsize=(10, 5)
)

ax.set_title(
    "Week 3 — Original Query vs LLM-Rewritten Query"
)
ax.set_ylabel("Mean metric value")
ax.set_xlabel("")

plt.xticks(
    rotation=25,
    ha="right"
)
plt.tight_layout()

fig_path = (
    RESULTS_DIR
    / "week3_bm25_rewrite_comparison.png"
)

plt.savefig(
    fig_path,
    dpi=180,
    bbox_inches="tight"
)

plt.show()

print("Largest NDCG improvements:")
display(
    per_query
    .sort_values(
        "delta_ndcg@10",
        ascending=False
    )
    [[
        "query",
        "rewritten_query",
        "judged_ndcg@10_original",
        "judged_ndcg@10_rewritten",
        "delta_ndcg@10"
    ]]
    .head(10)
)

print("Largest NDCG regressions:")
display(
    per_query
    .sort_values(
        "delta_ndcg@10",
        ascending=True
    )
    [[
        "query",
        "rewritten_query",
        "judged_ndcg@10_original",
        "judged_ndcg@10_rewritten",
        "delta_ndcg@10"
    ]]
    .head(10)
)

# Day 5 — Manual evaluation

After filling `week3_manual_validation_50.csv`, rerun the following cell.

It calculates:
- Intent Accuracy
- Core Product Accuracy
- Brand Accuracy
- Category Accuracy
- Attribute micro Precision / Recall / F1
- Mean rewrite quality score

In [ ]:
def norm_text(x):
    if pd.isna(x):
        return ""
    return re.sub(
        r"\s+",
        " ",
        str(x).strip().lower()
    )

def safe_json_loads(x, default):
    if pd.isna(x) or str(x).strip() == "":
        return default
    try:
        return json.loads(x)
    except Exception:
        return default

def pred_attribute_set(x):
    items = safe_json_loads(x, [])
    out = set()

    if isinstance(items, list):
        for item in items:
            if isinstance(item, dict):
                name = norm_text(
                    item.get("name")
                )
                value = norm_text(
                    item.get("value")
                )
                if name:
                    out.add((name, value))

    return out

def true_attribute_set(x):
    obj = safe_json_loads(x, {})
    out = set()

    if isinstance(obj, dict):
        for k, v in obj.items():
            out.add((
                norm_text(k),
                norm_text(v)
            ))

    elif isinstance(obj, list):
        for item in obj:
            if isinstance(item, dict):
                out.add((
                    norm_text(item.get("name")),
                    norm_text(item.get("value"))
                ))

    return out

completed_manual_path = (
    RESULTS_DIR
    / "week3_manual_validation_50.csv"
)

manual_eval = pd.read_csv(
    completed_manual_path
)

if (
    manual_eval["true_intent"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
    .all()
):
    print(
        "Ground-truth columns are blank. "
        "Fill week3_manual_validation_50.csv first."
    )
else:
    valid = manual_eval[
        manual_eval["true_intent"]
        .fillna("")
        .astype(str)
        .str.strip()
        != ""
    ].copy()

    def exact_acc(pred_col, true_col):
        return np.mean([
            norm_text(p) == norm_text(t)
            for p, t in zip(
                valid[pred_col],
                valid[true_col]
            )
        ])

    tp = 0
    fp = 0
    fn = 0

    for _, r in valid.iterrows():
        pred_set = pred_attribute_set(
            r["pred_attributes_json"]
        )
        true_set = true_attribute_set(
            r["true_attributes_json"]
        )

        tp += len(pred_set & true_set)
        fp += len(pred_set - true_set)
        fn += len(true_set - pred_set)

    attr_precision = (
        tp / (tp + fp)
        if (tp + fp)
        else 0.0
    )

    attr_recall = (
        tp / (tp + fn)
        if (tp + fn)
        else 0.0
    )

    attr_f1 = (
        2
        * attr_precision
        * attr_recall
        / (attr_precision + attr_recall)
        if (attr_precision + attr_recall)
        else 0.0
    )

    manual_summary = pd.DataFrame({
        "metric": [
            "annotated_queries",
            "intent_accuracy",
            "core_product_accuracy",
            "brand_accuracy",
            "category_accuracy",
            "attribute_precision_micro",
            "attribute_recall_micro",
            "attribute_f1_micro",
            "mean_rewrite_quality_0_to_2"
        ],
        "value": [
            len(valid),
            exact_acc(
                "pred_intent",
                "true_intent"
            ),
            exact_acc(
                "pred_core_product",
                "true_core_product"
            ),
            exact_acc(
                "pred_brand",
                "true_brand"
            ),
            exact_acc(
                "pred_category",
                "true_category"
            ),
            attr_precision,
            attr_recall,
            attr_f1,
            pd.to_numeric(
                valid["rewrite_quality_0_to_2"],
                errors="coerce"
            ).mean()
        ]
    })

    manual_summary.to_csv(
        RESULTS_DIR
        / "week3_manual_evaluation_summary.csv",
        index=False
    )

    display(manual_summary)

    print(
        "Attribute F1 target >= 0.75:",
        "MET" if attr_f1 >= 0.75 else "NOT YET MET"
    )

## 12. Export Week-3 result bundle

In [ ]:
import shutil
from pathlib import Path

readme_text = f"""
AMAZON SEARCH / LTR PROJECT — WEEK 3
====================================

MODEL
- {MODEL} via Ollama

PIPELINE
Raw Query
-> LLM Query Understanding
-> Structured JSON
-> Intent / Product / Attribute Extraction
-> Query Rewrite
-> BM25 Original-vs-Rewritten Evaluation

EVALUATION
1. Judged candidate set:
   - NDCG@10
   - MRR

2. Full 10k catalog:
   - Known-strong-relevant Recall@10
   - Known-strong-relevant Recall@50

TARGETS
- Expand toward 200 manually evaluated queries
- Attribute extraction F1 >= 0.75
- Improve retrieval quality through query rewriting

IMPORTANT
The Amazon 2018 dataset does not contain true search-query relevance judgments.
Week-1 relevance labels are heuristic proxies based on Amazon product relations.
"""

(
    RESULTS_DIR
    / "README_WEEK3.txt"
).write_text(
    readme_text.strip() + "\n",
    encoding="utf-8"
)

zip_path = shutil.make_archive(
    "amazon_week3_results",
    "zip",
    root_dir=RESULTS_DIR
)

print("Created:", zip_path)

print("Files:")
for p in sorted(RESULTS_DIR.iterdir()):
    print("-", p.name)

if running_in_colab():
    from google.colab import files
    files.download(zip_path)

## What a good Week-3 demo looks like

**Raw query:**  
`something comfortable for running outside when it rains`

**LLM understanding:**
```json
{
  "intent": "use_case_search",
  "core_product": "running shoes",
  "attributes": [
    {
      "name": "weather_resistance",
      "value": "waterproof",
      "explicit": false
    }
  ],
  "use_case": "running outdoors in rain",
  "rewritten_query": "waterproof outdoor running shoes"
}
```

Then show whether the rewrite improves:
- NDCG@10
- MRR
- full-catalog known-relevant Recall@10 / Recall@50

That gives the Week-3 story:

**LLM query understanding → structured representation → query rewrite → measurable retrieval change**

# Final Week-3 Report & Completion Check

This section creates a compact final scorecard and a Markdown report for your leader.

It uses the automated retrieval results immediately. If you have already filled the
manual-validation CSV, it will also include Intent Accuracy and Attribute F1.

In [ ]:
from datetime import datetime

def safe_metric(df, metric_name, metric_col='metric', value_col='value'):
    try:
        rows = df[df[metric_col] == metric_name]
        if len(rows) == 0:
            return np.nan
        return float(rows[value_col].iloc[0])
    except Exception:
        return np.nan

parse_success = safe_metric(auto_summary, 'parse_success_rate')
mean_latency = safe_metric(auto_summary, 'mean_latency_sec')

def delta_metric(name):
    try:
        return float(summary_delta.loc[summary_delta['metric'] == name, 'relative_delta_%'].iloc[0])
    except Exception:
        return np.nan

ndcg_delta = delta_metric('judged_ndcg@10')
mrr_delta = delta_metric('judged_mrr')
recall10_delta = delta_metric('known_relevant_recall@10')
recall50_delta = delta_metric('known_relevant_recall@50')

intent_accuracy = np.nan
attribute_f1 = np.nan
if 'manual_summary' in globals():
    intent_accuracy = safe_metric(manual_summary, 'intent_accuracy')
    attribute_f1 = safe_metric(manual_summary, 'attribute_f1_micro')

final_scorecard = pd.DataFrame({
    'metric': [
        'queries_run',
        'json_parse_success_rate',
        'mean_latency_sec',
        'manual_intent_accuracy',
        'manual_attribute_f1',
        'rewrite_ndcg10_relative_delta_%',
        'rewrite_mrr_relative_delta_%',
        'rewrite_recall10_relative_delta_%',
        'rewrite_recall50_relative_delta_%'
    ],
    'value': [
        len(structured),
        parse_success,
        mean_latency,
        intent_accuracy,
        attribute_f1,
        ndcg_delta,
        mrr_delta,
        recall10_delta,
        recall50_delta
    ]
})

final_scorecard.to_csv(RESULTS_DIR / 'week3_final_scorecard.csv', index=False)
display(final_scorecard)

def fnum(x, digits=4):
    return 'Not yet available' if pd.isna(x) else f'{x:.{digits}f}'

def fpct(x):
    return 'Not yet available' if pd.isna(x) else f'{x:.2f}%'

report_lines = [
    '# Amazon Search / Learning-to-Rank Project',
    '## Week 3 — LLM-Powered Query Understanding',
    '',
    '### 1. Objective',
    'Build a local Llama-3 query-understanding layer that converts natural-language product queries into structured intent/product/attribute representations and retrieval-friendly rewrites.',
    '',
    '### 2. Experimental Setup',
    f'- Model: {MODEL} via Ollama',
    f'- Week-1 catalog products: {len(products):,}',
    f'- Week-1 query-product pairs: {len(qpr):,}',
    f'- Fixed Week-2 test queries available: {len(query_table):,}',
    f'- Week-3 queries run: {len(structured):,}',
    '',
    '### 3. Pipeline',
    'Raw Query → Llama 3 → Structured JSON → Intent / Core Product / Attributes → Query Rewrite → BM25',
    '',
    '### 4. Technical Reliability',
    f'- JSON parse success rate: {fnum(parse_success)}',
    f'- Mean LLM latency: {fnum(mean_latency, 2)} sec/query',
    '',
    '### 5. Manual Query-Understanding Evaluation',
    f'- Intent accuracy: {fnum(intent_accuracy)}',
    f'- Attribute extraction F1: {fnum(attribute_f1)}',
    '- Roadmap target: Attribute F1 >= 0.75',
    '',
    '### 6. Query-Rewrite Retrieval Effect',
    f'- NDCG@10 relative change: {fpct(ndcg_delta)}',
    f'- MRR relative change: {fpct(mrr_delta)}',
    f'- Known-relevant Recall@10 relative change: {fpct(recall10_delta)}',
    f'- Known-relevant Recall@50 relative change: {fpct(recall50_delta)}',
    '',
    '### 7. Evaluation Note',
    'The Amazon 2018 source does not provide true search-query relevance judgments. Week-1 relevance labels are proxy labels derived from Amazon product relations. Full-catalog recall therefore measures recovery of known strong positives rather than treating every unjudged product as irrelevant.',
    '',
    '### 8. Week-4 Handoff',
    'Reuse rewritten_query, intent, core_product, attributes, and implicit requirements for Sentence-BERT / CLIP / FAISS semantic and multimodal retrieval.',
    '',
    f'Generated: {datetime.now().strftime("%Y-%m-%d %H:%M")}'
]

report_text = '\n'.join(report_lines)
report_path = RESULTS_DIR / 'Amazon_Week3_Final_Report.md'
report_path.write_text(report_text + '\n', encoding='utf-8')
print(report_text)


## Re-export the complete Week-3 bundle

Run this cell last. It includes the final scorecard/report in the ZIP.

In [ ]:
import shutil

final_zip = shutil.make_archive(
    'amazon_week3_full_results',
    'zip',
    root_dir=RESULTS_DIR
)

print('Created:', final_zip)
print('\nFinal deliverables:')
for p in sorted(RESULTS_DIR.iterdir()):
    print('-', p.name)

if running_in_colab():
    from google.colab import files
    files.download(final_zip)


# One-Week Completion Standard

Week 3 is complete when you have:

- **Day 1:** Ollama + Llama 3 running from Python with valid JSON.
- **Day 2:** intent classification + core product extraction working.
- **Day 3:** brand/category/attributes/price/use-case/implicit-demand extraction saved to CSV/Parquet.
- **Day 4:** original-query BM25 vs rewritten-query BM25 benchmark with NDCG@10, MRR, Recall@10, Recall@50 and error analysis.
- **Day 5:** 50 manually annotated queries, Intent Accuracy, Attribute Precision/Recall/F1, prompt-error review, and the final report.
- After the 50-query validation is stable, change `N_QUERIES = 200` and rerun for the roadmap-scale evaluation.

The final handoff to Week 4 is the structured/re-written query dataset.